## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    # 归一化：将像素值从[0,255]缩放到[0,1]
    x = x / 255.0
    x_test = x_test / 255.0
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([784, 128], stddev=0.01))  # 第1层权重
        self.b1 = tf.Variable(tf.zeros([128]))                             # 第1层偏置
        self.W2 = tf.Variable(tf.random.normal([128, 10], stddev=0.01))   # 第2层权重
        self.b2 = tf.Variable(tf.zeros([10]))                              # 第2层偏置

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        # 把28×28的图片压平成784维的向量
        x = tf.reshape(x, [-1, 784])
        
        # 线性变换 + ReLU激活函数
        h1 = tf.nn.relu(x @ self.W1 + self.b1)
        
        # 线性变换，输出logits
        logits = h1 @ self.W2 + self.b2
        
        return logits

model = myModel()
optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    
    # 用Adam优化器更新参数
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()

batch_size = 128  # 每次用128张图片训练（小批量梯度下降）

for epoch in range(50):
    indices = np.random.permutation(train_data[0].shape[0])
    
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]
    x_batch = tf.constant(train_data[0][batch_idx], dtype=tf.float32)
    y_batch = tf.constant(train_data[1][batch_idx], dtype=tf.int64)  # ← 加上 dtype=tf.int64
    loss, accuracy = train_one_step(model, optimizer, x_batch, y_batch)
    
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())

loss, accuracy = test(model,
                      tf.constant(test_data[0], dtype=tf.float32),
                      tf.constant(test_data[1], dtype=tf.int64))
print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.3014374 ; accuracy 0.09375
epoch 1 : loss 2.2922773 ; accuracy 0.1875
epoch 2 : loss 2.2783844 ; accuracy 0.46875
epoch 3 : loss 2.269883 ; accuracy 0.45833334
epoch 4 : loss 2.2502308 ; accuracy 0.59375
epoch 5 : loss 2.233113 ; accuracy 0.5833333
epoch 6 : loss 2.2267687 ; accuracy 0.48958334
epoch 7 : loss 2.2131732 ; accuracy 0.48958334
epoch 8 : loss 2.1774125 ; accuracy 0.5625
epoch 9 : loss 2.1435764 ; accuracy 0.6041667
epoch 10 : loss 2.0975468 ; accuracy 0.7395833
epoch 11 : loss 2.0745013 ; accuracy 0.65625
epoch 12 : loss 2.0708587 ; accuracy 0.6041667
epoch 13 : loss 2.0372505 ; accuracy 0.6666667
epoch 14 : loss 2.025728 ; accuracy 0.6458333
epoch 15 : loss 1.9273214 ; accuracy 0.6666667
epoch 16 : loss 1.8970112 ; accuracy 0.6979167
epoch 17 : loss 1.8541733 ; accuracy 0.7395833
epoch 18 : loss 1.8135792 ; accuracy 0.7395833
epoch 19 : loss 1.8332087 ; accuracy 0.6354167
epoch 20 : loss 1.7486719 ; accuracy 0.6666667
epoch 21 : loss 1.7179161 ; accuracy 